# Using `pyfwg` for direct morphing with `morph_epw_global`, `morph_epw_europe` and `morph_epw_cordex`

This notebook demonstrates the simplest and most direct way to use the `pyfwg` library: the `morph_epw_global`, `morph_epw_europe` and `morph_epw_cordex` functions.

These functions are designed as one-shot tools that provide full access to all of the Future Weather Generator's parameters without requiring the user to manage the multi-step `MorphingWorkflow` class. They handle parameter validation, temporary file management, and final file placement in a single call.

**Supported binaries:** Global 5.2.0, Europe EUR-11 3.2.0 and the other eight CORDEX-CMIP5 distributions 1.2.0. Older Global/Europe releases are outside the supported scope.

**New in pyfwg v0.4.0:** `pyfwg` also supports the **FWG Engine**, the shared engine of the latest distributions of the tool: **Global v5.x**, **Europe v3.x** and the **CORDEX regional** distributions. They are covered in the sections *Using `morph_epw_global` with the FWG Engine (Global v5.x)*, *Europe v3.x (FWG Engine)* and *Using `morph_epw_cordex`*.


## Configuration

The supported examples use Global 5.2.0, Europe EUR-11 3.2.0 and the other eight CORDEX-CMIP5 distributions 1.2.0. Older Global/Europe releases are outside the supported scope. Download JARs from the [FWG website](https://future-weather-generator.adai.pt/).

Paths are read from environment variables first, then an optional `fwg.local.json` in the project/working folder, then the defaults below. This local JSON file is excluded from Git. JARs can be on an external drive; keep it connected throughout a run. A path containing spaces needs no special escaping in a Python string.

- `FWG_JAR_DIR`: folder of regional JARs (default: project root); keep one version per domain.
- `FWG_GLOBAL_JAR_PATH`: optional full path to Global 5.2.0 if it is in another folder.
- `FWG_EPW_DIR`: real regional baselines named `<domain>.epw`. Global examples use `EUR-11.epw`. Retained validation samples are used if present; otherwise supply `epws/cordex`. Public sample links are in the regional notebook. Do not alter station coordinates to fit a region.
- `FWG_CACHE_DIR`: extracted climate data (default: `~/fwg_model_cache`). EUR-11's full ensemble alone contains about 11.3 GB before extraction overhead. Cache location is independent of the JAR folder.
- `FWG_OUTPUT_DIR`: results, Excel workbooks and diagnostics (default: `build/cordex-tutorial`).
- `FWG_JAVA`: optional Java executable; otherwise Java 17+ must be on PATH.

`RUN_MORPHING = False` prepares every example without starting Java. Set it to `True` to generate weather. Notebook outputs are cleared before distribution; generated files are kept for inspection.


In [ ]:
import json
import os
from pathlib import Path
from pyfwg import enable_console_logging

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyfwg" / "__init__.py").is_file()),
    Path.cwd(),
)
local_file = PROJECT_ROOT / "fwg.local.json"
local_paths = json.loads(local_file.read_text(encoding="utf-8")) if local_file.is_file() else {}

def configured_path(name, default):
    return Path(os.environ.get(name, local_paths.get(name, str(default)))).expanduser().resolve()

validation_epws = PROJECT_ROOT / "build/cordex-validation/epws"
default_epws = validation_epws if validation_epws.is_dir() else PROJECT_ROOT / "epws/cordex"
FWG_JAR_DIR = configured_path("FWG_JAR_DIR", PROJECT_ROOT)
FWG_GLOBAL_JAR_PATH = configured_path("FWG_GLOBAL_JAR_PATH", FWG_JAR_DIR / "FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar")
FWG_EPW_DIR = configured_path("FWG_EPW_DIR", default_epws)
FWG_CACHE_DIR = configured_path("FWG_CACHE_DIR", Path.home() / "fwg_model_cache")
FWG_OUTPUT_DIR = configured_path("FWG_OUTPUT_DIR", PROJECT_ROOT / "build/cordex-tutorial")
java_executable = os.environ.get("FWG_JAVA", local_paths.get("FWG_JAVA"))
if java_executable:
    java_path = Path(java_executable).expanduser().resolve()
    if not java_path.is_file():
        raise FileNotFoundError(f"Java executable not found: {java_path}")
    os.environ["PATH"] = str(java_path.parent) + os.pathsep + os.environ.get("PATH", "")
RUN_MORPHING = False
enable_console_logging()
print("Regional JARs:", FWG_JAR_DIR)


### Read a regional JAR and select compatible inputs

This helper reads the actual manifest, including renamed JARs. It selects one model by default, or every ensemble-eligible model with `ensemble=True`, then finds a scenario and period shared by the selection. Empty selections and ambiguous JAR folders fail with an explicit message. The returned dictionary uses the public `pyfwg` parameters; inspect or edit it before running an example.

Caches are separated by distribution ID **and version**, since the same GCM–RCM name can occur in different regions. All examples use `IDW`. `BI`/`1` needs a rectilinear geographic grid and is rejected for these rotated/projected regional grids. One distinct model disables ensemble mode; aliases containing underscores and hyphens refer to the same model. CORDEX Engine distributions do not implement UHI.


In [ ]:
from pyfwg import get_fwg_distribution_info

def regional_jar(domain):
    matches = []
    for jar in sorted(FWG_JAR_DIR.glob("*.jar")):
        info = get_fwg_distribution_info(str(jar))
        if info and info["framework"] == "CORDEX" and info["domain"] == domain:
            matches.append((jar, info))
    if len(matches) != 1:
        raise ValueError(f"Expected one {domain} JAR in {FWG_JAR_DIR}; found {len(matches)}. Check the folder and versions.")
    return matches[0]

def regional_inputs(domain, ensemble=False):
    jar, info = regional_jar(domain)
    models = info["ensemble_models"] if ensemble else info["models"][:1]
    if not models or (ensemble and len(models) < 2):
        raise ValueError(f"{domain}: need one model, or at least two eligible models for an ensemble.")
    details = info["model_details"]
    scenarios = [s for s in info["scenarios"] if all(s in details[m]["scenarios"] for m in models)]
    periods = [t for t in info["timeframes"] if all(t in details[m]["timeframes"] for m in models)]
    if not scenarios or not periods:
        raise ValueError(f"{domain}: selected models have no shared scenario or period.")
    epw = FWG_EPW_DIR / f"{domain}.epw"
    if not epw.is_file():
        raise FileNotFoundError(f"Supply a real baseline inside {domain} at {epw}, or set FWG_EPW_DIR.")
    options = dict(
        epw_paths=str(epw), fwg_jar_path=str(jar),
        fwg_rcm_pairs=list(models), fwg_create_ensemble=ensemble,
        fwg_scenarios=[scenarios[0]], fwg_timeframes=[periods[0]],
        fwg_interpolation_method_id="IDW", fwg_output_profile="MINIMAL",
        fwg_model_unzipped_folder=str(FWG_CACHE_DIR / info["distribution_id"] / info["version"]),
        fwg_extra_args={"threads": 1, "io_threads": 1, "save_technical_log": True},
        delete_temp_files=False,
    )
    return info, options


In [ ]:
def global_inputs():
    info = get_fwg_distribution_info(str(FWG_GLOBAL_JAR_PATH))
    if not info or info["framework"] != "CMIP" or info.get("domain"):
        raise ValueError(f"Supply a Global FWG Engine JAR at {FWG_GLOBAL_JAR_PATH}")
    epw = FWG_EPW_DIR / "EUR-11.epw"
    if not epw.is_file():
        raise FileNotFoundError(f"Supply the Global example baseline at {epw}")
    model = "CanESM5" if "CanESM5" in info["models"] else info["models"][0]
    details = info["model_details"][model]
    scenario = next(s for s in info["scenarios"] if s in details["scenarios"])
    period = next(t for t in info["timeframes"] if t in details["timeframes"])
    return info, dict(
        epw_paths=str(epw), fwg_jar_path=str(FWG_GLOBAL_JAR_PATH),
        fwg_gcms=[model], fwg_create_ensemble=False, fwg_add_uhi=False,
        fwg_scenarios=[scenario], fwg_timeframes=[int(period)],
        fwg_interpolation_method_id="IDW", fwg_output_profile="MINIMAL",
        fwg_use_multithreading=False,
        fwg_model_unzipped_folder=str(FWG_CACHE_DIR / info["distribution_id"] / info["version"]),
        fwg_extra_args={"threads": 1, "io_threads": 1, "save_technical_log": True},
        delete_temp_files=False,
    )


## Global 5.2.0: direct API

Inspect the actual catalog before selecting models. This example uses one model, one available SSP and one period. The API returns the collected future EPW/STAT paths. Engine distributions do not support UHI or LCZ morphing. Use `get_fwg_parameters_info()` for parameter descriptions.


In [ ]:
from pyfwg import morph_epw_global
info_global, global_options = global_inputs()
print(info_global["name"], info_global["version"])
print("Models:", info_global["models"])
print("Selection:", global_options)
if RUN_MORPHING:
    created_files = morph_epw_global(
        **global_options, output_dir=str(FWG_OUTPUT_DIR / "global-api"),
        temp_base_dir=str(FWG_OUTPUT_DIR / "temp-global-api"),
    )
    print(created_files)
else:
    print("Configuration only; set RUN_MORPHING=True to generate weather.")


### Europe v3.x (FWG Engine)

Europe EUR-11 v3.2.0 has passed real generation checks with a single model and its seven-member ensemble. It can be used with `morph_epw_europe` or `morph_epw_cordex`. Run Configuration and the regional helper above, then this section.

The example starts with one model, one shared scenario and one period. Set `ensemble=True` to use all eligible members, allowing at least 11.3 GB for climate data plus extraction overhead. Supply `EUR-11.epw` in `FWG_EPW_DIR` and set `RUN_MORPHING=True` to execute.


In [ ]:
from pyfwg import morph_epw_europe

info_europe, europe_options = regional_inputs("EUR-11", ensemble=False)
print(info_europe["name"], europe_options["fwg_rcm_pairs"], europe_options["fwg_scenarios"])
if RUN_MORPHING:
    created_files_europe_v3 = morph_epw_europe(
        **europe_options,
        output_dir=str(FWG_OUTPUT_DIR / "europe-api"),
        temp_base_dir=str(FWG_OUTPUT_DIR / "temp-europe-api"),
    )
    print(created_files_europe_v3)
    if not created_files_europe_v3:
        raise RuntimeError("No future weather files were collected; inspect the preserved engine diagnostics.")
else:
    print("Configuration inspected; set RUN_MORPHING=True to generate weather.")


Europe outputs and diagnostics remain in `FWG_OUTPUT_DIR` for inspection. The shared climate cache remains in `FWG_CACHE_DIR`.


In [ ]:
print("Europe API output:", FWG_OUTPUT_DIR / "europe-api")


## Using `morph_epw_cordex`

`morph_epw_cordex` accepts **all nine CORDEX-CMIP5 domains, including Europe EUR-11 v3+**. The installed JAR's manifest defines its region, models and scenarios. `CORDEX_REGIONS` is a fallback/reference table; use `get_fwg_distribution_info` for the catalog you actually installed. These distributions have no UHI/LCZ options.

The following table lists the domains. The [dedicated regional notebook](cordex_regions_example.ipynb) includes representative locations, tested versions and examples of all three interfaces.


In [ ]:
from pyfwg import morph_epw_cordex, CORDEX_REGIONS

for domain, region in CORDEX_REGIONS.items():
    print(domain, region["name"], region["scenarios"])


### Run the morphing process

This example uses the Cairo baseline `MNA-22.epw`, because MNA-22 has been tested with an actual station in that region. Change `DOMAIN` and supply the corresponding EPW to use another region. Java checks native spatial coverage. MNA-22 only offers RCP8.5; other regions offer different scenario subsets, which the helper reads from the JAR.

The default selects one model. Set `ensemble=True` for the complete eligible ensemble and `RUN_MORPHING=True` to run the generation cell.


In [ ]:
DOMAIN = "MNA-22"
info_cordex, cordex_options = regional_inputs(DOMAIN, ensemble=False)
print(info_cordex["name"], cordex_options["fwg_rcm_pairs"], cordex_options["fwg_scenarios"])
if RUN_MORPHING:
    created_files_cordex = morph_epw_cordex(
        **cordex_options,
        output_dir=str(FWG_OUTPUT_DIR / "direct" / DOMAIN),
        temp_base_dir=str(FWG_OUTPUT_DIR / "temp-direct" / DOMAIN),
    )
    print(created_files_cordex)
    if not created_files_cordex:
        raise RuntimeError("No future weather files were collected; inspect the preserved engine diagnostics.")
else:
    print("Configuration inspected; set RUN_MORPHING=True to generate weather.")


### Inspect results, SolarQC and disk usage

For EPW output, each selected scenario/period normally produces a future EPW and its STAT. The API returns their paths; workflows move and rename them into `final_output_dir`. Check that files exist: a failed Java run is logged and may leave an empty result list. A preview alone does not establish that the EPW passes the engine's checks.

These examples keep `delete_temp_files=False` and technical logs. For SolarQC rejection, inspect `<temp_base_dir>/<EPW stem>/metadata/solarqc_preprocessing_failure.txt` and the `logs/application` folder. During the 2026-10-28 integration checks, New Delhi and Suvarnabhumi samples were rejected; unmodified public Mumbai and Singapore samples passed. Changing the JAR or station coordinates is not a repair for an unsuitable baseline. `MINIMAL` reduces auxiliary output; mandatory diagnostics can still be written.

After reviewing results and diagnostics, you can remove the corresponding temporary directory and the **specific distribution/version cache** when no run is using it. `delete_temp_files=True` removes per-EPW temporary files after a successful run; it does not clear a separately configured shared cache. No cleanup cell in these regional examples removes your results or caches automatically. An iterator reuses caches but does not evict previous regions: with limited space, run one region at a time and clean its cache before continuing.

The downloaded TMYx test samples cover 2009–2023, whereas the CORDEX climate reference period is 1976–2005. FWG reports this mismatch. These are software integration examples; choose and assess the baseline/reference period for your study. See the [CORDEX regional notebook](cordex_regions_example.ipynb) for the nine-region validation matrix and source links.


In [ ]:
print("Regional API output:", FWG_OUTPUT_DIR / "direct" / DOMAIN)
